# Teste do Tacotron 2 Original da NVIDIA

In [1]:
import os
# Desligar a GPU no nivel do sistema para forcar 100% CPU no torch.hub
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import torch
import IPython.display as ipd
import numpy as np
import sys
from pathlib import Path

# Forcar o uso exclusivo de CPU
device = torch.device('cpu')
print(f"Usando device: {device}")

Usando device: cpu


In [ ]:
# 1. Carregar a arquitetura do Tacotron 2 da NVIDIA via PyTorch Hub
tacotron2 = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_tacotron2', pretrained=False, trust_repo=True)

# 2. Carregar os pesos locais (o modelo pre-treinado que vc tem)
# Como o notebook esta em notebooks/, voltamos uma pasta (../) para acessar local_weight_models
checkpoint_path = '../local_weight_models/tacotron2/nvidia_tacotron2pyt_fp32_20190427'
checkpoint = torch.load(checkpoint_path, map_location='cpu')

# Pegar o state_dict e remover 'module.' se necessario
state_dict = checkpoint.get('state_dict', checkpoint)
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

tacotron2.load_state_dict(state_dict)
tacotron2 = tacotron2.to(device).eval()
print("Pesos do Tacotron 2 carregados com sucesso!")

Using cache found in /home/richard/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub
/home/richard/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub/PyTorch/Classification/ConvNets/image_classification/models/common.py:13: UserWarning: pytorch_quantization module not found, quantization will not be available
  warnings.warn(
/home/richard/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub/PyTorch/Classification/ConvNets/image_classification/models/efficientnet.py:17: UserWarning: pytorch_quantization module not found, quantization will not be available
  warnings.warn(
/home/richard/project/ml2_final_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: Error(s) in loading state_dict for Tacotron2:
	Missing key(s) in state_dict: "embedding.weight". 
	Unexpected key(s) in state_dict: "transcript_embedding.weight", "vae_gst.ref_encoder.convs.0.weight", "vae_gst.ref_encoder.convs.0.bias", "vae_gst.ref_encoder.convs.0.conv.weight", "vae_gst.ref_encoder.convs.0.conv.bias", "vae_gst.ref_encoder.convs.1.weight", "vae_gst.ref_encoder.convs.1.bias", "vae_gst.ref_encoder.convs.2.weight", "vae_gst.ref_encoder.convs.2.bias", "vae_gst.ref_encoder.convs.3.weight", "vae_gst.ref_encoder.convs.3.bias", "vae_gst.ref_encoder.convs.4.weight", "vae_gst.ref_encoder.convs.4.bias", "vae_gst.ref_encoder.convs.5.weight", "vae_gst.ref_encoder.convs.5.bias", "vae_gst.ref_encoder.bns.0.weight", "vae_gst.ref_encoder.bns.0.bias", "vae_gst.ref_encoder.bns.0.running_mean", "vae_gst.ref_encoder.bns.0.running_var", "vae_gst.ref_encoder.bns.0.num_batches_tracked", "vae_gst.ref_encoder.bns.1.weight", "vae_gst.ref_encoder.bns.1.bias", "vae_gst.ref_encoder.bn

In [ ]:
# 3. Carregar o Vocoder (WaveGlow) da NVIDIA sem pesos via Hub
waveglow = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_waveglow', pretrained=False, trust_repo=True)

# Carregar os pesos localmente de forma segura na CPU
waveglow_ckpt_path = '../local_weight_models/waveglow/nvidia_waveglowpyt_fp32_20190427'
waveglow_ckpt = torch.load(waveglow_ckpt_path, map_location='cpu')
state_dict = waveglow_ckpt.get('state_dict', waveglow_ckpt)
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

waveglow.load_state_dict(state_dict)
waveglow = waveglow.remove_weightnorm(waveglow)
waveglow = waveglow.to(device).eval()
print('WaveGlow carregado com sucesso puramente em CPU!')


Using cache found in /home/richard/.cache/torch/hub/NVIDIA_DeepLearningExamples_torchhub


WaveGlow carregado com sucesso puramente em CPU!


In [ ]:
# 4. Text Processing
# Para usar os cleaners originais (english_cleaners), vamos importar diretamente
# do repositorio que o torch.hub acabou de clonar na sua pasta de cache.
hub_dir = Path(torch.hub.get_dir()) / 'NVIDIA_DeepLearningExamples_torchhub' / 'PyTorch' / 'SpeechSynthesis' / 'Tacotron2'
if str(hub_dir) not in sys.path:
    sys.path.append(str(hub_dir))

from tacotron2.text import text_to_sequence

texto = "Hello! This is a test of the pre-trained NVIDIA Tacotron 2 model."
print(f"Texto: {texto}")

# Converter texto em sequencia numerica de tokens
sequence = np.array(text_to_sequence(texto, ['english_cleaners']))[None, :]
# Assegurar que a sequencia seja movida para a CPU
sequence = torch.autograd.Variable(torch.from_numpy(sequence)).to(device).long()

Texto: Hello! This is a test of the pre-trained NVIDIA Tacotron 2 model.


In [ ]:
# 5. Geracao e Audio
with torch.no_grad():
    # Passar texto pelo Tacotron2
    mel_outputs_postnet, mel_lengths, alignments = tacotron2.infer(sequence, torch.LongTensor([sequence.size(1)]).to(device))
    
    # Passar Espectrograma Mel pelo WaveGlow para gerar audio
    audio = waveglow.infer(mel_outputs_postnet, sigma=0.666)

audio_numpy = audio[0].data.cpu().numpy()

# 6. Tocar o Audio no Jupyter
ipd.Audio(audio_numpy, rate=22050)